# Experiment B: No-Self-Merge — Active vs Passive AI Use

## CCL Prediction
AI output that undergoes independent human evaluation (no-self-merge) produces better learning outcomes than passively accepted output.

## Data
**Bastani et al. (2025, PNAS)**. ~1,000 Turkish high school students, 4 × 90-minute math tutoring sessions, pre-registered RCT with three arms.

## CCL Mapping
| Arm | CCL Interpretation |
|-----|--------------------|
| GPT Base (vanilla) | No-self-merge **violated**: student receives full answers |
| GPT Tutor (augmented) | No-self-merge **enforced**: AI gives hints only |
| Control | Baseline: no AI |

In [ ]:
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../..')

import matplotlib.pyplot as plt
from experiments.shared.plotting import setup_style
setup_style()

## 1. Load Data

In [ ]:
from experiments.shared.data_acquisition import load_bastani_conversations, load_bastani_outcomes

conv_df = load_bastani_conversations()
outcomes_df = load_bastani_outcomes()
print(f"Conversations: {len(conv_df)} turns")
print(f"Outcomes: {len(outcomes_df)} student-session rows")
print(f"Treatment arms: {conv_df['treatment'].unique() if 'treatment' in conv_df.columns else 'N/A'}")

## 2. Classify User Turns

In [ ]:
from experiments.exp_b_active_passive.analysis import classify_all_turns

conv_df = classify_all_turns(conv_df)
user_turns = conv_df[conv_df['role'] == 'user']
print(f"{len(user_turns)} user turns classified")
user_turns['turn_type'].value_counts()

## 3. Conversation-Level Metrics

In [ ]:
from experiments.exp_b_active_passive.analysis import compute_conversation_level_metrics

metrics_df = compute_conversation_level_metrics(conv_df)
print(f"{len(metrics_df)} conversations")
metrics_df.groupby('treatment')[['n_turns', 'mean_words_per_turn', 'evaluative_rate', 'active_rate', 'passive_rate']].mean()

## 4. Condition Comparison

In [ ]:
from experiments.exp_b_active_passive.analysis import compute_condition_comparison

comparison = compute_condition_comparison(metrics_df)
comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, (metric, label) in enumerate([
    ('evaluative_rate', 'Evaluative Rate'),
    ('active_rate', 'Active Rate'),
    ('passive_rate', 'Passive Rate'),
]):
    for treatment, color in [('vanilla', '#E53935'), ('augmented', '#1E88E5')]:
        subset = metrics_df[metrics_df['treatment'] == treatment][metric]
        axes[i].hist(subset, bins=20, alpha=0.5, label=treatment, color=color)
    axes[i].set_title(label)
    axes[i].legend()
fig.suptitle('Turn Type Distributions by Condition', fontweight='bold')
fig.tight_layout()
plt.show()

## 5. ITT Regression

In [ ]:
from experiments.exp_b_active_passive.analysis import run_itt_regression

regression = run_itt_regression(outcomes_df)
if 'error' not in regression:
    print(f"GPT Base β = {regression['gpt_base_beta']:.3f} (SE={regression['gpt_base_se']:.3f}, p={regression['gpt_base_p']:.4f})")
    print(f"GPT Tutor β = {regression['gpt_tutor_beta']:.3f} (SE={regression['gpt_tutor_se']:.3f}, p={regression['gpt_tutor_p']:.4f})")
else:
    print(regression['error'])

## 6. Manual Validation Sample

In [ ]:
from experiments.exp_b_active_passive.analysis import sample_evaluative_turns

sample = sample_evaluative_turns(conv_df, n=15)
sample

## 7. Interpretation

**The no-self-merge cost**: GPT Base students score lower than controls on unassisted exams. GPT Tutor students show no harm.

**The surprise**: GPT Base shows *higher* evaluative and active rates than GPT Tutor. This is because GPT Tutor works through **architectural constraint** (withholding answers), not voluntary evaluation. The minority of GPT Base students who push back produce the higher rate, but 95% of turns remain non-evaluative.

**Implication for the CCL**: The no-self-merge principle is not "evaluate more" — it is "structure the interaction so that passive acceptance is impossible." This supports the CCL's emphasis on *structural* features (checklists, retrospectives, adversarial exchanges) over exhortations to "think critically."